# Load libraries

In [ ]:
import os
import sys
import gc

import numpy as np
import warnings
import pandas as pd
import json

from collections import Counter
from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=pd.errors.SettingWithCopyWarning)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

In [ ]:
import json
import pandas as pd
import numpy as np
import random

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import spacy
# load the spacy pipline model
nlp = spacy.load("en_core_web_lg")

# Read data

## List

In [ ]:
%%time
test = json.load((open('/kaggle/input/pii-detection-removal-from-educational-data/train.json')))

len(test)

In [ ]:
test[0].keys()

## Pandas

In [ ]:
%%time
df = pd.read_json('/kaggle/input/pii-detection-removal-from-educational-data/train.json')

df.shape

In [ ]:
df.columns

# Custom functions

In [ ]:
# Function to find documents that contain a specified label
def find_documents_with_label(df, target_label):
    
    document_numbers = []
    
    # Iterate over the DataFrame rows
    for index, row in df.iterrows():
        # Check if target_label is in the labels list for the current row
        if target_label in row['labels']:
            # If so, append the document number to the list
            document_numbers.append(row['document'])
    
    return document_numbers

In [ ]:
#Function to convert a single row from the dataframe to SpaCy format
def convert_to_spacy_format(text, tokens, labels, trailing_whitespace):
    ents = []  # To store entity dictionaries
    start = 0  # Position tracker for the start of each token in the text
    
    for i, (token, label, space) in enumerate(zip(tokens, labels, trailing_whitespace)):
        if label.startswith('B-') or label.startswith('I-'):
            label_type = label[2:]  # Extract entity type from label
            token_start = text.find(token, start)  # Find the start index of the token in text
            token_end = token_start + len(token)  # Calculate the end index of the token
            
            # If it's a 'B-' label or the first 'I-' label following non-matching or 'O' labels, start a new entity
            if label.startswith('B-') or (label.startswith('I-') and (i == 0 or not labels[i-1].endswith(label_type))):
                ents.append({"start": token_start, "end": token_end, "label": label_type})
            # If it's an 'I-' label continuing an entity, extend the last entity's end index
            elif label.startswith('I-') and ents and ents[-1]["label"] == label_type:
                ents[-1]["end"] = token_end
            
            start = token_end + (1 if space == 'True' else 0)  # Update start position for next token
    
    return [{"text": text, "ents": ents, "title": None}]

In [ ]:
def encode_labels(df):
    df["unique_labels"] = df["labels"].apply(lambda x: list(set(
        [l.split('-')[1] for l in x if l != 'O']
         )))
    # add 1-hot encoding
    from sklearn.preprocessing import MultiLabelBinarizer

    mlb = MultiLabelBinarizer()
    one_hot_encoded = mlb.fit_transform(df['unique_labels'])
    one_hot_df = pd.DataFrame(one_hot_encoded, columns=mlb.classes_)
    df = pd.concat([df, one_hot_df], axis=1)
    
    # add 'OTHER' column
    df['OTHER'] = df['unique_labels'].apply(lambda x: 1 if len(x) == 0 else 0)
    
    return df, list(mlb.classes_) + ['OTHER']



# Target counts

In [ ]:
# Get all labels present in the dataframe
# Step 1: Flatten the labels column
all_labels = [label for sublist in df['labels'] for label in sublist]

# Step 2: Use Counter to count occurrences of each label
label_counts = Counter(all_labels)

label_counts

In [ ]:
df, label_classes = encode_labels(df)

for col in label_classes:
    print(f'{col}: {df[col].sum()}')

In [ ]:
df.head()

# Document length

In [ ]:
df['essay_length'] = df['full_text'].apply(lambda x: len(x.split()))
df['tokens_length'] = df['tokens'].apply(lambda x: len(x))
df['num_labels'] = df['labels'].apply(lambda x: sum(label != 'O' for label in x))

df.shape

In [ ]:
print(df['essay_length'].describe())

sns.displot(df['essay_length'])

In [ ]:
print(df['tokens_length'].describe())

sns.displot(df['tokens_length'])

In [ ]:
print(df[df['num_labels']!=0]['num_labels'].describe())

sns.displot(df[df['num_labels']!=0]['num_labels'])

In [ ]:
df['num_labels'].value_counts().sort_index()

# View Examples

In [ ]:
df[df['num_labels']==34]

In [ ]:
# target_label = 'O'
target_label = 'B-URL_PERSONAL'

document_numbers = find_documents_with_label(df, target_label)

docnum = random.sample(document_numbers, 1)

# num = 2769
num = docnum[0]

# spacy color options for manual render
options = {"colors": {"NAME_STUDENT": "#748CAB", 
                      "URL_PERSONAL": "#FFFC31", 
                      "ID_NUM": "#E94F37", 
                      "EMAIL": "#F8B195", 
                      "STREET_ADDRESS": "#BDBF09", 
                      "PHONE_NUM": "#D96C06", 
                      "USERNAME": "#2292A4"}}

# Filtering dataframe for row via document ID
filtered_row = df[df['document'] == num]

# Assuming there's only one such row, getting the `full_text` content
display_text = filtered_row['full_text'].values[0] if not filtered_row.empty else "Document not found"
display_labels = filtered_row['labels'].values[0] if not filtered_row.empty else "No Labels found"
trailing_whitespace = filtered_row['trailing_whitespace'].values[0] if not filtered_row.empty else []
tokens = filtered_row['tokens'].values[0] if not filtered_row.empty else []

# Cleanup new line for SpaCy rendering
display_text = display_text.replace("\n\n", "\r\n")

# Get labels from dataframe and convert to SpaCy format
ex = convert_to_spacy_format(display_text, tokens, display_labels, trailing_whitespace)


print(docnum)

In [ ]:
doc = 3202

row = df[df['document']==doc]

tokens = row['tokens'].values[0]
labels = row['labels'].values[0]


for token, label in zip(tokens, labels):
    print(f"{token}: {label}")

In [ ]:
# Display labels from dataframe
spacy.displacy.render(ex, style="ent", manual=True, jupyter=True, options=options)

# Explore

In [ ]:
random.sample(docs, 10)

In [ ]:
docs = []

for row in df.itertuples():
    
    if 'design thinking' in getattr(row,'full_text'):
        docs.append(getattr(row,'document'))
        
        
    
len(docs)

In [ ]:
urls = {}

for row in df.itertuples():
    
    for token in getattr(row,'tokens'):
        
        if "://" in token:
            if getattr(row,'document') not in urls.keys():
                urls[getattr(row,'document')] = [token]
            else:
                urls[getattr(row,'document')].append(token)

len(urls)

In [ ]:
urls